# Centralised LSTM fh=1 modified

This notebook reports `avg_normalised_loss`, `avg_actual_loss`, saves base/reconciled/naive forecasts, and evaluates MAE, MASE, RMSSE.

For CLSTM, there is one centralised model, so early stopping is checked once for the global training loop.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from clstm import (
    Datacollector,
    run_centralised,
    compute_metrics_from_dicts,
    make_reconciliation_frames,
    evaluate_reconciliation_results,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [2]:
# -------------------------
# Parameter setup
# -------------------------
#args_path = 'C:/Users/n12553263/yjzPyR/Datasets/GEFCOM2017/GEF2017_univarivate.csv'
args_path = "E:/yjz/Datasets/AusGrid2013/AusGrid2013_univariate.csv" #'E:/yjz/Datasets/GEFCOM2017/Gef2017_univarivate.csv'
args_fh  = 0
args_lags = 48
args_INPUT, args_HIDDEN, args_LAYERS, args_DROP = 1, 64, 2, 0.1
args_EPOCHS, args_BATCH_SIZE, args_LR = 100 + args_fh*50, 256, 1e-3
args_TRAIN_RATIO = 0.8
args_MODEL_NAME = 'clstm'
args_truncated = None
args_SCALING_MODE = 'per_partition'  # 'global' or 'per_partition'
args_DROP_BOUNDARY_GAP = True

# Early stopping settings
args_EARLY_STOP_ENABLED = True
args_EARLY_STOP_TOL = 1e-5
args_MIN_EPOCHS = 20 + args_fh*50

raw_df = pd.read_csv(args_path)
series_ids = sorted(raw_df['partition_id'].dropna().unique())
dict_ = Datacollector(raw_df, lags=args_lags, ts=series_ids, fh=args_fh)
df = pd.concat([dict_[key] for key in dict_.keys()]).dropna()
df['ds'] = pd.to_datetime(df['ds'], format='mixed')
if args_truncated is not None:
    df = df[df['ds'] >= args_truncated].reset_index(drop=True)
df = df.reset_index(drop=True)

parts = sorted(df['partition_id'].unique())
lag_cols_reversed = list(reversed(sorted([c for c in df.columns if c.startswith('lags_')], key=lambda x: int(x.split('_')[1]))))
forecast_horizon = ['y'] + sorted([c for c in df.columns if c.startswith('post_')], key=lambda x: int(x.split('_')[1]))
df2 = df[['unique_id', 'partition_id', 'ds'] + lag_cols_reversed + forecast_horizon].copy()

print(df2.head())
print(f'num partitions: {len(parts)}')
print(f'Scaling mode: {args_SCALING_MODE} | drop_boundary_gap: {args_DROP_BOUNDARY_GAP}')
print(f'X shape: {df2[lag_cols_reversed].shape} | y shape: {df2[forecast_horizon].shape}')

100%|████████████████████████████████████████████████████████████████████████████████| 400/400 [00:14<00:00, 27.70it/s]


  unique_id  partition_id                  ds  lags_48  lags_47  lags_46  \
0        GC             0 2012-07-02 00:00:00  102.719   83.000   75.919   
1        GC             0 2012-07-02 00:30:00   83.000   75.919   75.508   
2        GC             0 2012-07-02 01:00:00   75.919   75.508   65.438   
3        GC             0 2012-07-02 01:30:00   75.508   65.438   63.545   
4        GC             0 2012-07-02 02:00:00   65.438   63.545   63.436   

   lags_45  lags_44  lags_43  lags_42  ...   lags_9   lags_8   lags_7  \
0   75.508   65.438   63.545   63.436  ...  244.433  237.004  225.504   
1   65.438   63.545   63.436   61.718  ...  237.004  225.504  217.168   
2   63.545   63.436   61.718   56.149  ...  225.504  217.168  208.276   
3   63.436   61.718   56.149   61.600  ...  217.168  208.276  187.653   
4   61.718   56.149   61.600   63.447  ...  208.276  187.653  171.450   

    lags_6   lags_5   lags_4   lags_3   lags_2   lags_1        y  
0  217.168  208.276  187.653  171.450

In [3]:
df2

,unique_id,partition_id,ds,lags_48,lags_47,lags_46,lags_45,lags_44,lags_43,lags_42,...,lags_9,lags_8,lags_7,lags_6,lags_5,lags_4,lags_3,lags_2,lags_1,y
0,GC,0,2012-07-02 00:00:00,102.719,83.000,75.919,75.508,65.438,63.545,63.436,...,244.433,237.004,225.504,217.168,208.276,187.653,171.450,145.573,120.626,102.174
1,GC,0,2012-07-02 00:30:00,83.000,75.919,75.508,65.438,63.545,63.436,61.718,...,237.004,225.504,217.168,208.276,187.653,171.450,145.573,120.626,102.174,88.861
2,GC,0,2012-07-02 01:00:00,75.919,75.508,65.438,63.545,63.436,61.718,56.149,...,225.504,217.168,208.276,187.653,171.450,145.573,120.626,102.174,88.861,79.038
3,GC,0,2012-07-02 01:30:00,75.508,65.438,63.545,63.436,61.718,56.149,61.600,...,217.168,208.276,187.653,171.450,145.573,120.626,102.174,88.861,79.038,71.534
4,GC,0,2012-07-02 02:00:00,65.438,63.545,63.436,61.718,56.149,61.600,63.447,...,208.276,187.653,171.450,145.573,120.626,102.174,88.861,79.038,71.534,66.693
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6988795,GC/2330/85,399,2013-06-30 21:30:00,0.296,0.990,1.929,1.671,1.530,0.330,0.288,...,0.371,0.318,0.414,1.194,0.341,0.299,0.313,0.340,0.435,1.165
6988796,GC/2330/85,399,2013-06-30 22:00:00,0.990,1.929,1.671,1.530,0.330,0.288,0.925,...,0.318,0.414,1.194,0.341,0.299,0.313,0.340,0.435,1.165,0.307
6988797,GC/2330/85,399,2013-06-30 22:30:00,1.929,1.671,1.530,0.330,0.288,0.925,0.096,...,0.414,1.194,0.341,0.299,0.313,0.340,0.435,1.165,0.307,0.305
6988798,GC/2330/85,399,2013-06-30 23:00:00,1.671,1.530,0.330,0.288,0.925,0.096,0.088,...,1.194,0.341,0.299,0.313,0.340,0.435,1.165,0.307,0.305,0.316


In [4]:
%%time
results = run_centralised(
    df=df2,
    lag_cols_reversed=lag_cols_reversed,
    forecast_horizon=forecast_horizon,
    input_size=args_INPUT,
    hidden_size=args_HIDDEN,
    num_layers=args_LAYERS,
    dropout=args_DROP,
    batch_size=args_BATCH_SIZE,
    epochs=args_EPOCHS,
    lr=args_LR,
    train_ratio=args_TRAIN_RATIO,
    partition_col='partition_id',
    device=None,
    clip_grad=1.0,
    verbose=True,
    lags=args_lags,
    fh=args_fh,
    time_col='ds',
    scaling_mode=args_SCALING_MODE,
    drop_boundary_gap=args_DROP_BOUNDARY_GAP,
    early_stop_enabled=args_EARLY_STOP_ENABLED,
    early_stop_tol=args_EARLY_STOP_TOL,
    min_epochs=args_MIN_EPOCHS,
    checkpoint_path=f'fh{args_fh+1}_cen_lstm.pt',
)

parts = results['parts']
train_idx = results['train_idx']
test_idx = results['test_idx']
dict_pred = results['dict_pred']
dict_true = results['dict_true']
dict_tr_pred = results['dict_tr_pred']
dict_tr_true = results['dict_tr_true']
dict_naive = results['dict_naive']

print('Device:', results['device'])
print('Stopped early:', results['stopped_early'])
print('Stop epoch:', results['stop_epoch'])
print('Final avg_normalised_loss:', results['final_avg_normalised_loss'])
print('Final avg_actual_loss:', results['final_avg_actual_loss'])
pd.DataFrame(results['round_logs']).tail()

[Centralised] epoch 001/100 | avg_normalised_loss=0.419243 | avg_actual_loss=0.204053 | delta=nan | stopped=False
[Centralised] epoch 002/100 | avg_normalised_loss=0.412281 | avg_actual_loss=0.185287 | delta=0.00696192 | stopped=False
[Centralised] epoch 003/100 | avg_normalised_loss=0.411193 | avg_actual_loss=0.186450 | delta=0.00108824 | stopped=False
[Centralised] epoch 004/100 | avg_normalised_loss=0.404302 | avg_actual_loss=0.184793 | delta=0.00689106 | stopped=False
[Centralised] epoch 005/100 | avg_normalised_loss=0.402857 | avg_actual_loss=0.183450 | delta=0.00144453 | stopped=False
[Centralised] epoch 006/100 | avg_normalised_loss=0.401421 | avg_actual_loss=0.184177 | delta=0.0014369 | stopped=False
[Centralised] epoch 007/100 | avg_normalised_loss=0.400712 | avg_actual_loss=0.195099 | delta=0.000708533 | stopped=False
[Centralised] epoch 008/100 | avg_normalised_loss=0.397671 | avg_actual_loss=0.172043 | delta=0.0030414 | stopped=False
[Centralised] epoch 009/100 | avg_normal

,epoch,avg_normalised_loss,avg_actual_loss,normalised_loss_delta,stopped
95,96,0.367160,0.163678,0.001903,False
96,97,0.366715,0.157157,0.000445,False
97,98,0.367128,0.161399,0.000413,False
98,99,0.366368,0.162302,0.000760,False
99,100,0.366898,0.158438,0.000529,False


In [5]:
# Base model and Naive baseline metrics before reconciliation
base_metrics = compute_metrics_from_dicts(dict_true, dict_pred, dict_train_true=dict_tr_true, h_idx=0)
naive_metrics = compute_metrics_from_dicts(dict_true, dict_naive, dict_train_true=dict_tr_true, h_idx=0)
display(base_metrics.tail())
display(naive_metrics.tail())

,partition_id,n_test,MAE,MASE,RMSSE
396,396,3504,0.240930,1.143667,0.980056
397,397,3504,0.229164,1.049820,1.022619
398,398,3504,0.072528,1.006764,0.948951
399,399,3504,0.253959,0.942607,0.808231
400,Overall,1401600,0.164786,1.085357,0.993099


,partition_id,n_test,MAE,MASE,RMSSE
396,396,3504,0.286559,1.360264,1.217229
397,397,3504,0.257102,1.177806,1.201794
398,398,3504,0.077455,1.075168,1.024470
399,399,3504,0.283192,1.051108,0.961046
400,Overall,1401600,0.185562,1.167698,1.133205


In [6]:
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace

#path = 'C:/Users/n12553263/yjzPyR/Datasets/GEFCOM2017/'
path = "E:/yjz/Datasets/AusGrid2013/"#'E:/yjz/Datasets/GEFCOM2017/'
S = pd.read_csv(f'{path}AusGrid_S.csv')
tags = pd.read_pickle(f'{path}tags.pkl')

reconciliation_methods = [
    BottomUp(),
    MinTrace(method='mint_shrink'),
    MinTrace(method='wls_var'),
    MinTrace(method='ols'),
    MinTrace(method='wls_struct'),
]
reconcilers = HierarchicalReconciliation(reconcilers=reconciliation_methods)

recon_results = {}
train_frames = {}
test_frames = {}
for H_IDX in range(args_fh + 1):
    y_df, y_hat_df, train_frame, test_frame = make_reconciliation_frames(
        df=df2,
        train_idx=train_idx,
        test_idx=test_idx,
        parts=parts,
        dict_tr_pred=dict_tr_pred,
        dict_pred=dict_pred,
        forecast_horizon=forecast_horizon,
        base_model_name=args_MODEL_NAME,
        horizon_idx=H_IDX,
    )
    train_frames[H_IDX] = train_frame
    test_frames[H_IDX] = test_frame
    recon_results[H_IDX] = reconcilers.reconcile(Y_hat_df=y_hat_df, S=S, tags=tags, Y_df=y_df)

recon_results[0].head()

C:\Users\qifei\AppData\Local\Temp\ipykernel_25684\1264891743.py:35: DeprecationWarning: The 'S' parameter is deprecated and will be removed in a future version. Please use 'S_df' instead.
  recon_results[H_IDX] = reconcilers.reconcile(Y_hat_df=y_hat_df, S=S, tags=tags, Y_df=y_df)


,unique_id,ds,y,clstm,clstm/BottomUp,clstm/MinTrace_method-mint_shrink,clstm/MinTrace_method-wls_var,clstm/MinTrace_method-ols,clstm/MinTrace_method-wls_struct
0,GC,2013-04-19 00:00:00,78.031,72.416588,73.134460,72.468078,73.490988,72.441899,73.288905
1,GC,2013-04-19 00:30:00,61.409,70.598723,67.984952,70.519160,68.943892,70.581929,69.362313
2,GC,2013-04-19 01:00:00,57.268,59.388005,57.657920,58.379590,57.954730,59.362771,58.261325
3,GC,2013-04-19 01:30:00,57.180,55.670369,56.914969,55.384843,56.178118,55.673816,56.037206
4,GC,2013-04-19 02:00:00,51.572,55.105345,54.999621,54.979752,55.145994,55.108589,55.147274


In [7]:
len(y_hat_df['unique_id'].unique())

400

In [8]:
args_fh

0

In [9]:
# Save base, reconciled and naive forecasts; evaluate MAE, MASE and RMSSE
output_prefix = f'fh{args_fh+1}_cen'
eval_outputs = evaluate_reconciliation_results(
    recon_results=recon_results,
    train_frames=train_frames,
    test_frames=test_frames,
    forecast_horizon=forecast_horizon,
    base_model_name=args_MODEL_NAME,
    output_prefix=output_prefix,
    approach='centralised',
    round_logs=results['round_logs'],
    timings=results['timings'],
)

forecast_table = eval_outputs['forecast_table']
per_series_metrics = eval_outputs['per_series_metrics']
metrics_by_level = eval_outputs['metrics_by_level']
overall_metrics = eval_outputs['overall_metrics']
timing_df = eval_outputs['timing_df']
output_paths = eval_outputs['output_paths']

print('Saved outputs:')
for k, v in output_paths.items():
    print(f'  {k}: {v}')

metrics_by_level

Saved outputs:
  forecasts: fh1_cen_forecasts.csv
  per_series_metrics: fh1_cen_per_series_metrics.csv
  metrics_by_level: fh1_cen_metrics_by_level.csv
  overall_metrics: fh1_cen_overall_metrics.csv
  round_logs: fh1_cen_round_logs.csv
  timing: fh1_cen_timing.csv


,level,role,method,MAE,MASE,RMSSE,n_series,approach
0,1,top,base,5.056800,0.825376,0.732379,1,centralised
1,1,top,bu,5.710614,0.932092,0.914479,1,centralised
2,1,top,mint_ols,5.045286,0.823496,0.731009,1,centralised
3,1,top,mint_shrinkage,4.897508,0.799376,0.723733,1,centralised
4,1,top,mint_var,5.247796,0.856550,0.819176,1,centralised
5,1,top,naive,7.763219,1.267120,1.198684,1,centralised
6,1,top,wls_structure,5.059202,0.825768,0.771629,1,centralised
7,2,middle,base,0.238354,1.069268,0.994117,100,centralised
8,2,middle,bu,0.235152,1.059534,0.990023,100,centralised
9,2,middle,mint_ols,0.239717,1.096228,0.987918,100,centralised


In [10]:
overall_metrics

,method,MAE,MASE,RMSSE,n_series,approach
0,base,0.164786,1.085357,0.993099,400,centralised
1,bu,0.165620,1.083190,0.992531,400,centralised
2,mint_ols,0.168237,1.125728,0.993538,400,centralised
3,mint_shrinkage,0.164626,1.094923,0.986279,400,centralised
4,mint_var,0.164849,1.087637,0.988650,400,centralised
5,naive,0.185562,1.167698,1.133205,400,centralised
6,wls_structure,0.164717,1.093527,0.987656,400,centralised


In [11]:
timing_df

,module,seconds
0,training_sec,37683.900358
1,prediction_sec,123.826510
2,total_sec,37862.526995
